# Histopathology Backbone Pretraining (5-class)

Pretrain MobileNetV3-Large on 5-class histopathology images. At the end we save:
- **Backbone weights** (for transfer to oral binary/5-class)
- Full model checkpoint (optional)
- Class index mapping

**Data**: `~/dev/oral-images-dataset/histopathology-images` — 5 folders = 5 labels: mdoscc, normal, osmf, pdoscc, wdoscc.

## 1. Configuration

In [1]:
import os
import json
from pathlib import Path
from datetime import datetime

HISTOLOGY_DIR = Path(os.path.expanduser("~/dev/oral-images-dataset/histopathology-images"))
NOTEBOOK_DIR = Path(".").resolve()
RUNS_DIR = NOTEBOOK_DIR / "results" / "histology_pretrain"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
run_name = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = RUNS_DIR / run_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)

CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "num_epochs": 50,
    "patience": 12,
    "val_split": 0.15,
    "num_workers": 0,
    "seed": 42,
    "lr": 1e-4,
    "dropout_rate": 0.2,
    "label_smoothing": 0.05,
}

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump({**CONFIG, "image_size": list(CONFIG["image_size"]), "run_name": run_name}, f, indent=2)

print(f"Histology dir: {HISTOLOGY_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"TensorBoard (all runs): tensorboard --logdir={RUNS_DIR}")
print(f"TensorBoard (this run): tensorboard --logdir={OUTPUT_DIR}")

Histology dir: /Users/alokdwivedi/dev/oral-images-dataset/histopathology-images
Output: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/run_20260214_080629
TensorBoard (all runs): tensorboard --logdir=/Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain
TensorBoard (this run): tensorboard --logdir=/Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/run_20260214_080629


## 2. Device and imports

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALB = True
except ImportError:
    HAS_ALB = False

try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TB = True
except ImportError:
    HAS_TB = False

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

Using Apple Silicon GPU (MPS)


## 3. Load histopathology data (5 classes)

In [3]:
def collect_histology_paths_labels(root_dir, exts=(".jpg", ".jpeg", ".png")):
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"Histology dir not found: {root}")
    # Class names = folder names (sorted for stable indices)
    class_names = sorted([d.name for d in root.iterdir() if d.is_dir() and not d.name.startswith(".")])
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    paths, labels = [], []
    ext_set = {e.lower() for e in exts}
    for cls in class_names:
        d = root / cls
        if not d.is_dir():
            continue
        # Recursive: images may be in subfolders (e.g. mdoscc/o-3-00-30/*.png)
        for p in d.rglob("*"):
            if p.is_file() and p.suffix.lower() in ext_set:
                paths.append(str(p))
                labels.append(class_to_idx[cls])
    return paths, labels, class_names, class_to_idx


all_paths, all_labels, CLASS_NAMES, CLASS_TO_IDX = collect_histology_paths_labels(HISTOLOGY_DIR)

print(f"Total images: {len(all_paths)}")
print(f"Classes (sorted): {CLASS_NAMES}")
for c in CLASS_NAMES:
    n = sum(1 for lb in all_labels if CLASS_TO_IDX[c] == lb)
    print(f"  {c}: {n}")

Total images: 10228
Classes (sorted): ['mdoscc', 'normal', 'osmf', 'pdoscc', 'wdoscc']
  mdoscc: 2699
  normal: 1045
  osmf: 2095
  pdoscc: 1599
  wdoscc: 2790


## 4. Train/val split

In [4]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=CONFIG["val_split"],
    random_state=CONFIG["seed"],
    stratify=all_labels,
)

print(f"Train: {len(train_paths)}, Val: {len(val_paths)}")

Train: 8693, Val: 1535


## 5. Augmentation and dataset

In [5]:
def get_transforms():
    if HAS_ALB:
        train_t = A.Compose([
            A.Resize(CONFIG["image_size"][0], CONFIG["image_size"][1]),
            A.RandomRotate90(p=0.4),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
            A.HueSaturationValue(hue_shift_limit=12, sat_shift_limit=15, val_shift_limit=12, p=0.4),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
        val_t = A.Compose([
            A.Resize(CONFIG["image_size"][0], CONFIG["image_size"][1]),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
        return train_t, val_t, True
    import torchvision.transforms as T
    t = T.Compose([
        T.Resize(CONFIG["image_size"]),
        T.RandomHorizontalFlip(),
        T.RandomRotation(15),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    v = T.Compose([
        T.Resize(CONFIG["image_size"]),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return t, v, False


class HistologyDataset(Dataset):
    def __init__(self, paths, labels, transform=None, use_albu=True):
        self.paths = paths
        self.labels = labels
        self.transform = transform
        self.use_albu = use_albu

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            if self.use_albu:
                img = self.transform(image=np.array(img))["image"]
            else:
                img = self.transform(img)
        return img, label


train_t, val_t, use_albu = get_transforms()
train_ds = HistologyDataset(train_paths, train_labels, transform=train_t, use_albu=use_albu)
val_ds = HistologyDataset(val_paths, val_labels, transform=val_t, use_albu=use_albu)

train_loader = DataLoader(
    train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
    num_workers=CONFIG["num_workers"], pin_memory=False,
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"],
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 272, Val batches: 48


/Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/venv_pytorch/lib/python3.11/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


## 6. Model: MobileNetV3-Large + 5-class head

In [6]:
NUM_CLASSES = len(CLASS_NAMES)


class HistologyMobileNetV3(nn.Module):
    """MobileNetV3-Large with 5-class head (960 -> 256 -> num_classes)."""

    def __init__(self, num_classes=5, dropout_rate=0.2):
        super().__init__()
        try:
            self.backbone = models.mobilenet_v3_large(
                weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
            )
        except AttributeError:
            self.backbone = models.mobilenet_v3_large(pretrained=True)
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(960, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.backbone.features(x)
        x = self.classifier(x)
        return x


model = HistologyMobileNetV3(num_classes=NUM_CLASSES, dropout_rate=CONFIG["dropout_rate"]).to(device)
total = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total:,}")

Model parameters: 3,219,253


## 7. Class weights and training setup

In [7]:
classes = np.unique(train_labels)
weights = compute_class_weight("balanced", classes=classes, y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=CONFIG.get("label_smoothing", 0.0),
)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG["num_epochs"], eta_min=1e-6
)

writer = SummaryWriter(log_dir=str(OUTPUT_DIR / "tb_logs")) if HAS_TB else None

## 8. Train and validate (multiclass)

In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device).long()
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, 100.0 * correct / n


@torch.no_grad()
def validate_multiclass(model, loader, criterion, device, num_classes):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    probs_list, targets_list = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device).long()
        out = model(x)
        loss = criterion(out, y)
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        n += x.size(0)
        probs_list.append(torch.softmax(out, 1).cpu().numpy())
        targets_list.append(y.cpu().numpy())
    probs = np.concatenate(probs_list)
    targets = np.concatenate(targets_list)
    acc = 100.0 * correct / n
    try:
        auc = roc_auc_score(targets, probs, multi_class="ovr", average="weighted")
    except Exception:
        auc = 0.0
    return acc, auc, total_loss / n

In [9]:
best_acc, best_auc, best_state, patience_ctr = 0.0, 0.0, None, 0

for epoch in range(CONFIG["num_epochs"]):
    loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_acc, val_auc, val_loss = validate_multiclass(model, val_loader, criterion, device, NUM_CLASSES)
    scheduler.step()
    lr = optimizer.param_groups[0]["lr"]

    if writer:
        writer.add_scalar("loss/train", loss, epoch)
        writer.add_scalar("acc/train", train_acc, epoch)
        writer.add_scalar("acc/val", val_acc, epoch)
        writer.add_scalar("auc/val", val_auc, epoch)
        writer.add_scalar("lr", lr, epoch)
        writer.flush()

    if val_auc > best_auc:
        best_auc = val_auc
        best_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Ep {epoch+1:3d}/{CONFIG['num_epochs']}  loss={loss:.4f}  "
              f"train_acc={train_acc:.2f}%  val_acc={val_acc:.2f}%  val_auc={val_auc:.4f}  lr={lr:.2e}")

    if patience_ctr >= CONFIG["patience"]:
        print(f"Early stop at epoch {epoch+1}")
        break

if writer:
    writer.close()
if best_state:
    model.load_state_dict(best_state)

print(f"\nBest: val_acc={best_acc:.2f}%, val_auc={best_auc:.4f}")

Ep   1/50  loss=1.0294  train_acc=59.52%  val_acc=76.29%  val_auc=0.9378  lr=9.99e-05


Ep   5/50  loss=0.5231  train_acc=86.60%  val_acc=90.94%  val_auc=0.9883  lr=9.76e-05


Ep  10/50  loss=0.3903  train_acc=93.40%  val_acc=93.36%  val_auc=0.9951  lr=9.05e-05


Ep  15/50  loss=0.3477  train_acc=95.43%  val_acc=95.11%  val_auc=0.9946  lr=7.96e-05


Ep  20/50  loss=0.3279  train_acc=96.39%  val_acc=94.98%  val_auc=0.9948  lr=6.58e-05


Ep  25/50  loss=0.3076  train_acc=97.61%  val_acc=95.24%  val_auc=0.9946  lr=5.05e-05


Early stop at epoch 28

Best: val_acc=95.44%, val_auc=0.9962


## 9. Save backbone and metadata

In [ ]:
model.eval()

# Full model (optional)
torch.save(model.state_dict(), OUTPUT_DIR / "models" / "full_model.pth")

# Backbone only (for transfer)
backbone_state = {k: v for k, v in model.state_dict().items() if k.startswith("backbone.")}
torch.save(backbone_state, OUTPUT_DIR / "backbone_state_dict.pth")

# Class mapping for reference
meta = {
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "num_classes": NUM_CLASSES,
    "best_val_acc": float(best_acc),
    "best_val_auc": float(best_auc),
}
with open(OUTPUT_DIR / "class_mapping.json", "w") as f:
    json.dump(meta, f, indent=2)

# Copy backbone and class_mapping to RUNS_DIR so transfer notebook finds latest
import shutil
for fname in ("backbone_state_dict.pth", "class_mapping.json"):
    src = OUTPUT_DIR / fname
    if src.exists():
        shutil.copy2(src, RUNS_DIR / fname)

print(f"Saved (run folder):")
print(f"  {OUTPUT_DIR / 'models' / 'full_model.pth'}")
print(f"  {OUTPUT_DIR / 'backbone_state_dict.pth'}")
print(f"  {OUTPUT_DIR / 'class_mapping.json'}")
print(f"Latest for transfer: {RUNS_DIR / 'backbone_state_dict.pth'}")

Saved (run folder):
  /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/run_20260214_080629/models/full_model.pth
  /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/run_20260214_080629/backbone_state_dict.pth
  /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/run_20260214_080629/class_mapping.json
Latest for transfer: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/backbone_state_dict.pth
